## Import Functions

In [1]:
from __future__ import annotations

import whisper
import soundfile as sf
import torch
from nltk.cluster import cosine_distance
from torch import Tensor
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer

import phonemizer
from phonemizer.backend import EspeakBackend
from sentence_transformers import SentenceTransformer
import os

import soundfile as sf
from nltk.tokenize import word_tokenize # Tokenizers divide strings into lists of substrings
import yaml

from dataclasses import dataclass

from models import *
from utils import *
from text_utils import TextCleaner

from Utils.PLBERT.util import load_plbert

import IPython.display as ipd

from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule
# os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = r"C:\Program Files\eSpeak NG\libespeak-ng.dll"  # <-- adjust if different
torch.manual_seed(0) # Fixes starting point of random seed for torch
torch.backends.cudnn.benchmark = False # Fix convolution algorithm
torch.backends.cudnn.deterministic = True # Only use deterministic algorithms

%cd ..

/home/yanis/Documents/University/Bachelorarbeit/StyleTTS2


#### Optimizer imports

In [2]:
from dataclasses import dataclass, field
from typing import Any, Optional, Union

from numpy.typing import NDArray

import logging
from typing import Any, Optional, Type

import logging
from abc import ABC, abstractmethod
from typing import Any, Iterable, Optional, Type

import numpy as np
from numpy.typing import NDArray
from torch import Tensor

import phonemizer

import numpy as np
from numpy.typing import NDArray
from pymoo.algorithms.base.genetic import GeneticAlgorithm
from pymoo.core.evaluator import Evaluator
from pymoo.core.population import Population
from pymoo.core.problem import Problem
from pymoo.core.termination import NoTermination
from pymoo.problems.static import StaticProblem

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling

## Create Helper Functions


In [3]:
def length_to_mask(lengths: Tensor) -> Tensor:
    mask = torch.arange(lengths.max())  # Creates a Vector [0,1,2,3,...,x], where x = biggest value in lengths
    mask = mask.unsqueeze(0)  # Creates a Matrix [1,x] from Vector [x]
    mask = mask.expand(lengths.shape[0], -1)  # Expands the matrix from [1,x] to [y,x], where y = number of elements in lengths
    mask = mask.type_as(lengths)  # Assign mask the same type as lengths
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))  # gt = greater than, compares each value from lengths to a row of values in mask; unsqueeze = splits vector lengths into vectors of size 1
    return mask  # returns a mask of shape (batch_size, max_length) where mask[i, j] = 1 if j < lengths[i] and mask[i, j] = 0 otherwise.

In [4]:
def addNumbersPattern(to_change: Tensor, reference: Tensor, pattern: list[int]) -> Tensor:

    diff = reference.size(-1) - to_change.size(-1)

    if diff < 0:
        print("Reference is smaller then whats to be changed")
        return to_change

    padding = torch.as_tensor(pattern, device=to_change.device, dtype=to_change.dtype)[torch.arange(diff, device=to_change.device) % len(pattern)]

    # Match batch dimensions, same logic as torch.full
    for _ in range(to_change.dim() - 1):
        padding = padding.unsqueeze(0)
    padding = padding.expand(*to_change.shape[:-1], diff)

    # Concatenate along last dimension
    to_change = torch.cat([to_change, padding], dim=-1)

    return to_change

In [25]:
def interpolateWithVector(ground_truth: Tensor, target: Tensor, alpha: Tensor) -> Tensor:
    """
    Phoneme-level interpolation between ground_truth and target embeddings.

    ground_truth: (B, D, T)
    target:       (B, D, T)
    alpha:        (T,) torch.Tensor
                  alpha[t] = 0 → choose ground truth
                  alpha[t] = 1 → choose target
    """

    assert ground_truth.shape == target.shape, "ground_truth and target must have same shape"
    B, D, T = ground_truth.shape

    # Ensure alpha has correct length
    assert alpha.shape[-1] == T, f"alpha length {alpha.shape[-1]} != T={T}"

    # Move alpha to correct device + dtype
    alpha = alpha.to(ground_truth.device, ground_truth.dtype)

    # Reshape for broadcasting: (1, 1, T)
    alpha = alpha.view(1, 1, T)

    # Interpolate
    mixed = (1.0 - alpha) * ground_truth + alpha * target
    return mixed

In [6]:
def interpolateWithScalar(a: Tensor, b: Tensor, alpha: float) -> Tensor:
    return (1 - alpha) * a + alpha * b


## Classes

In [7]:
class AutomaticSpeechRecognitionModel:
    def __init__(self, model_name="tiny"):
        self.model = whisper.load_model(model_name)

    def analyzeAudio(self, wav: torch.Tensor) -> tuple[dict, float]:
        result = self.model.transcribe(audio=wav)

        total_logprob = 0.0
        total_tokens = 0
        for seg in result["segments"]:
            total_logprob += seg["avg_logprob"] * len(seg["tokens"])
            total_tokens += len(seg["tokens"])

        avg_logprob = total_logprob / total_tokens if total_tokens > 0 else float("nan")
        return result, avg_logprob

In [8]:
class TextEmbeddingModel:
    """
    Wrapper around a SentenceTransformer model for computing
    text embeddings and cosine distances.
    """

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-mpnet-base-v2",
        device: Optional[str] = None,
    ):
        """
        model_name: HuggingFace model id, e.g. "sentence-transformers/all-mpnet-base-v2"
        device: "cpu", "cuda", or None to let sentence-transformers decide
        """
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, text: str | list[str]) -> torch.Tensor:
        """
        Encodes a single string or list of strings.
        Returns a tensor of shape:
          - [D] for a single string
          - [N, D] for a list of N strings

        Embeddings are L2-normalized for cosine distance.
        """
        emb = self.model.encode(
            text,
            convert_to_tensor=True,
            normalize_embeddings=True,
        )
        return emb

    @staticmethod
    def cosine_distance(a: torch.Tensor, b: torch.Tensor) -> float:
        """
        Cosine distance in [0, ~1] for real sentence embeddings.
          0 = identical
          1 ≈ maximally different
        """
        if a.ndim == 1:
            a = a.unsqueeze(0)
        if b.ndim == 1:
            b = b.unsqueeze(0)

        cos_sim = F.cosine_similarity(a, b).item()
        return 1.0 - cos_sim

    def distance_between_texts(self, text1: str, text2: str) -> float:
        """
        Convenience helper: encode two texts and return cosine distance.
        """
        e1 = self.encode(text1)
        e2 = self.encode(text2)
        return self.cosine_distance(e1, e2)

In [9]:
@dataclass
class OptimizerCandidate:
    """A candidate solution found by the optimizer."""

    solution: Optional[NDArray]
    fitness: Union[tuple[float, ...], float, list[float]]

    # Additional data that might be used.
    data: Optional[Any] = field(default=None)
    parents: Optional[list[OptimizerCandidate]] = field(default=None)

    def __post_init__(self) -> None:
        """Post init processing of data."""
        if isinstance(self.fitness, float):
            self.fitness: tuple[float, ...] = (self.fitness,)
        elif isinstance(self.fitness, list):
            self.fitness: tuple[float, ...] = tuple(self.fitness)

In [10]:
class Optimizer(ABC):
    """An abstract optimizer class."""

    # Standard elements (if applicable).
    _best_candidates: list[OptimizerCandidate]  # The current best candidate.
    _previous_best: list[OptimizerCandidate]  # The previous best solution candidate.
    _x_current: NDArray  # The current solution.
    _fitness: tuple[NDArray, ...]  # The current fitness.
    _bounds: tuple[int, int]  # Solution bounds.
    _n_var: int  # Solution size.

    # Always exist.
    _optimizer_type: Type[Any]
    _num_objectives: int

    def __init__(self, num_objectives: int) -> None:
        """
        Init default values.

        :param num_objectives: The number of objectives.
        """
        self._optimizer_type = type(self)
        self._num_objectives = num_objectives

    @abstractmethod
    def update(self) -> None:
        """
        Generate a new population.
        """
        ...

    @abstractmethod
    def get_x_current(self) -> NDArray:
        """
        Return the current population in specific format.

        :return: The population as array.
        """
        ...

    def assign_fitness(self, fitness: Iterable[NDArray], *data: Optional[Iterable[Any]]) -> None:
        """
        Assign fitness to the current population and extract the best individual using pareto frontier.

        :param fitness: The fitness to assign.
        :param data: Additional data for the candidates.
        """
        logging.info(f"Assigning fitness to {self.__class__.__name__}")
        # Format fitness into tuple if it is list or singular item.
        fitness = [f.numpy() if isinstance(f, Tensor) else f for f in fitness]
        fitness = tuple(fitness)

        assert (
            len(fitness) == self._num_objectives
        ), f"Error: {len(fitness)} Fitness values found, {self._num_objectives} expected."

        self._fitness = fitness

        """Since we can have an arbitrary amount of metrics we extract best candidates using a pareto frontier."""
        new_metrics = np.ascontiguousarray(
            fitness
        ).T  # Metrics are rows, instances are columns -> we transpose.
        old_metrics = np.ascontiguousarray([cand.fitness for cand in self._best_candidates])
        metrics = np.vstack((new_metrics, old_metrics))

        new_data: list[Any] = [None] * new_metrics.shape[0] if data is None else list(zip(*data))
        data = tuple(new_data + [cand.data for cand in self._best_candidates])

        solutions = np.vstack(
            (self._x_current, np.array([cand.solution for cand in self._best_candidates]))
        )

        sorted_indices = metrics.sum(1).argsort()
        for i in range(metrics.shape[0]):
            n = sorted_indices.shape[0]
            on_pareto = np.ones(n, dtype=bool)
            if i >= n:
                break
            on_pareto[i + 1 : n] = (
                metrics[sorted_indices][i + 1 :] <= metrics[sorted_indices][i]
            ).all(axis=1) & (metrics[sorted_indices][i + 1 :] < metrics[sorted_indices][i]).any(
                axis=1
            )
            sorted_indices = sorted_indices[on_pareto[:n]]

        candidates = []
        for index in sorted_indices:
            candidates.append(
                OptimizerCandidate(
                    solution=solutions[index], fitness=metrics[index], data=data[index]
                )
            )
        self._previous_best = self._best_candidates
        self._best_candidates = candidates

    def reset(self) -> None:
        """Reset the learner to the default."""
        self._x_current = np.random.uniform(
            low=self._bounds[0], high=self._bounds[1], size=self._x_current.shape
        )
        self._best_candidates = [
            OptimizerCandidate(self._x_current[0], [np.inf] * self._num_objectives)
        ]
        self._previous_best = self._best_candidates.copy()

    @property
    def best_candidates(self) -> list[OptimizerCandidate]:
        """
        Get the best candidates so far (if more than one it is a pareto frontier).

        :return: The candidate.
        """
        return self._best_candidates

    @property
    def previous_best(self) -> list[OptimizerCandidate]:
        """
        Get the previous best candidates.

        :return: The candidate.
        """
        return self._previous_best

    @property
    def optimizer_type(self) -> Type[Any]:
        """
        Get the type of the optimizer.

        :returns: The type.
        """
        return self._optimizer_type

    @property
    def n_var(self) -> int:
        """
        Get size of genome for optimizer.

        :returns: The size of the genome.
        """
        return self._n_var

    def _clip_to_bounds(self, element: NDArray) -> NDArray:
        """
        Clip array values to the range of bounds.

        :param element: The array to be clipped.
        :returns: The clipped array.
        """
        element = np.maximum(element, self._bounds[0])
        element = np.minimum(element, self._bounds[1])
        return element

In [11]:
class PymooOptimizer(Optimizer):
    """A Learner class for easy Pymoo integration"""

    _pymoo_algo: GeneticAlgorithm
    _problem: Problem
    _pop_current: Population
    _bounds: tuple[int, int]
    _shape: tuple[int, ...]

    _params: dict[str, Any]
    _algorithm: Type[GeneticAlgorithm]

    def __init__(
        self,
        bounds: tuple[int, int],
        algorithm: Type[GeneticAlgorithm],
        algo_params: dict[str, Any],
        num_objectives: int,
        solution_shape: tuple[int, ...],
    ) -> None:
        """
        Initialize the genetic learner.

        :param bounds: Bounds for the optimizer.
        :param algorithm: The pymoo Algorithm.
        :param algo_params: Parameters for the pymoo Algorithm.
        :param num_objectives: The number of objectives the learner can handle.
        :param solution_shape: The shape of the solution arrays.
        """
        super().__init__(num_objectives)
        """Initialize Constants."""
        self._params = algo_params
        self._algorithm = algorithm
        self._bounds = bounds

        """Initialize optimization problem and initial solutions."""
        self.update_problem(solution_shape)
        self._optimizer_type = type(self._pymoo_algo)

    def update(self) -> None:
        """
        Generate a new population.
        """
        logging.info("Sampling new population...")
        static = StaticProblem(self._problem, F=np.column_stack(self._fitness))
        Evaluator().eval(static, self._pop_current)
        self._pymoo_algo.tell(self._pop_current)

        self._pop_current = self._pymoo_algo.ask()
        self._x_current = self._clip_to_bounds(self._pop_current.get("X"))

    def get_x_current(self) -> NDArray:
        """
        Return the current population in a specific format.

        :return: The currently best genome.
        """
        return self._x_current.reshape((self._x_current.shape[0], *self._shape))

    def update_problem(
        self, solution_shape: tuple[int, ...], sampling: Optional[NDArray] = None
    ) -> None:
        """
        Change problem shape of optimization.

        :param solution_shape: The new solution shape.
        :param sampling: An initial solution array if available.
        """
        if sampling is not None:
            assert np.prod(solution_shape) == np.prod(
                sampling.shape[1:]
            ), f"ERROR: sampling shape {sampling.shape[1:]}, does not conform to solution size {solution_shape}."
            x0 = sampling.reshape(sampling.shape[0], -1)
            x0 = self._clip_to_bounds(x0)
            self._params["sampling"] = x0

        self._shape = solution_shape
        self._n_var = int(np.prod(solution_shape))
        self._pymoo_algo = self._algorithm(**self._params, save_history=True)

        self._problem = Problem(
            n_var=self._n_var,
            n_obj=self._num_objectives,
            xl=self._bounds[0],
            xu=self._bounds[1],
            vtype=float,
        )
        self._pymoo_algo.setup(self._problem, termination=NoTermination())
        self.reset()

    def reset(self) -> None:
        """Resets the optimizer."""
        self._pop_current = self._pymoo_algo.ask()
        self._x_current = self._clip_to_bounds(self._pop_current.get("X"))

        self._best_candidates = [
            OptimizerCandidate(
                solution=np.random.uniform(
                    high=self._bounds[0], low=self._bounds[1], size=self._n_var
                ),
                fitness=[np.inf] * self._num_objectives,
            )
        ]
        self._previous_best = self._best_candidates.copy()

    @property
    def best_solutions_reshaped(self) -> list[NDArray]:
        """
        Get the best solutions in correct shape.

        :return: The solutions.
        """
        return [
            c.solution.reshape(self._shape) for c in self._best_candidates if c.solution is not None
        ]

In [12]:
@dataclass
class InferenceResult:
    h_text: Tensor
    h_aligned: Tensor
    f0_pred: Tensor
    a_pred: Tensor
    n_pred: Tensor
    style_vector_prosodic: Tensor

    def save(self, folder: str):

        os.makedirs("outputs/latent/"+folder, exist_ok=True)

        # Iterate through all fields of the dataclass
        for name, value in self.__dict__.items():
            if isinstance(value, Tensor):
                path = os.path.join("outputs/latent/"+folder, f"{name}.pt")
                torch.save(value, path)
                print(f"✅ Saved {name} -> {path}")
            else:
                print(f"⚠️ Skipping {name} (not a tensor)")

In [13]:
class StyleTTS2:

    h_text: Tensor
    h_aligned: Tensor
    f0_pred: Tensor
    a_pred: Tensor
    n_pred: Tensor
    style_vector_prosodic: Tensor

    def __init__(self):

        # Splits words into phonemes (symbols that represent how words are pronounced)
        self.model = None
        self.params = None
        self.sampler = None

        self.global_phonemizer = phonemizer.backend.EspeakBackend(
            language='en-us',
            preserve_punctuation=True,  # Keeps Punctuation such as , . ? !
            with_stress=True  # Adds stress marks to vowels
        )

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        self.textcleaner = TextCleaner()  # Lowercasing & trimming, expanding numbers & symbols, handling punctuation, phoneme conversion, tokenization

    def load_models(self, yml_path="Models/LJSpeech/config.yml"):
        config = yaml.safe_load(open(yml_path))  # YAML File with model settings and pretrained checkpoints (ASR, F0, PL-BERT)

        # load pretrained ASR (Automatic Speech Recognition) model
        ASR_config = config.get('ASR_config', False)  # YAML config that describes the model’s structure
        ASR_path = config.get('ASR_path', False)  # Checkpoint File
        text_aligner = load_ASR_models(ASR_path, ASR_config)  # Load PyTorch model

        # load pretrained F0 model (Extracts Pitch Features from Audio, How Pitch Changes over time)
        F0_path = config.get('F0_path', False)  # YAML config that describes the model’s structure
        pitch_extractor = load_F0_models(F0_path)

        # load BERT model (encodes input text with prosodic cues)
        BERT_path = config.get('PLBERT_dir', False)  # YAML config that describes the model’s structure
        plbert = load_plbert(BERT_path)

        self.model = build_model(
            recursive_munch(config['model_params']),  # Allows attribute-style access to keys of model_params,
            text_aligner,  # Automatic Speech Recognition model
            pitch_extractor,  # F0 model
            plbert  # BERT model
        )

        _ = [self.model[key].eval() for key in self.model]
        _ = [self.model[key].to(self.device) for key in self.model]

        params_whole = torch.load("Models/LJSpeech/epoch_2nd_00100.pth", map_location='cpu', weights_only=False)
        self.params = params_whole['net']

    def load_checkpoints(self):
        for key in self.model:
            if key in self.params:
                try:
                    self.model[key].load_state_dict(self.params[key])
                except:
                    from collections import OrderedDict
                    state_dict = self.params[key]
                    new_state_dict = OrderedDict()
                    for k, v in state_dict.items():
                        name = k[7:]  # remove `module.`
                        new_state_dict[name] = v
                    # load params
                    self.model[key].load_state_dict(new_state_dict, strict=False)
        #             except:
        #                 _load(params[key], model[key])
        _ = [self.model[key].eval() for key in self.model]

    def sample_diffusion(self):
        self.sampler = DiffusionSampler(
            self.model.diffusion.diffusion,
            sampler=ADPM2Sampler(),
            sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),  # empirical parameters
            clamp=False
        )

    # Turns text to tensor with token ID
    def preprocessText(self, text: str) -> Tensor:
        # 1. Preprocessing Text
        text = text.strip()  # Removes whitespaces from beginning and end of string
        text = text.replace('"', '')  # removes " to prevent unpredictable behavior

        # 2. Text -> Phoneme
        phonemes = self.global_phonemizer.phonemize([text])  # text -> list of phoneme
        phonemes = word_tokenize(phonemes[0])  # Split into individual tokens
        phonemes = ' '.join(phonemes)  # Join tokens together, split by a empty space

        # 3. Phoneme -> Token ID
        tokens = self.textcleaner(phonemes)  # Look up numeric ID per phoneme
        tokens.insert(0, 0)  # Insert leading 0 to mark start

        # 4. Token ID -> PyTorch Tensor
        tokens = torch.LongTensor(tokens).to(self.device).unsqueeze(0)  # Converts numeric ID to PyTorch Tensor

        return tokens

    def predictDuration(self, bert_encoder_with_style: Tensor, input_lengths: Tensor) -> Tensor:

        # Duration Predictor, frames per phoneme
        d_pred, _ = self.model.predictor.lstm(bert_encoder_with_style)  # Model temporal dependencies between phonemes, LSTM = RNN
        d_pred = self.model.predictor.duration_proj(d_pred)  # Predict how long each phoneme lasts
        d_pred = torch.sigmoid(d_pred).sum(axis=-1)  # Sum of duration prediction -> Result: Prediction of frame duration
        d_pred = torch.round(d_pred.squeeze()).clamp(min=1)  # Convert duration prediction into integers, add clamp to ensure that each phoneme has at least one frame
        d_pred[-1] += 5  # Makes last phoneme last 5 frames longer, to ensure it not being cut off too fast

        # Creates predicted alignment matrix between text (phonemes) and audio frames
        a_pred = torch.zeros(input_lengths, int(d_pred.sum().data))  # Initializes a matrix with sizes: [# of Phonemes (input_lengths)] x [Sum of total predicted frames]
        current_frame = 0
        for i in range(a_pred.size(0)):  # Iterates over phoneme
            a_pred[i, current_frame:current_frame + int(d_pred[i].data)] = 1  # Changes for row-i (the i-th phoneme) all the values from current_frame to current_frame + int(d_pred[i].data) to 1
            current_frame += int(d_pred[i].data)  # Move current_frame to new first start

        return a_pred

    @torch.no_grad()
    def computeStyleVector(self, noise: Tensor, h_bert: Tensor, embedding_scale: int, diffusion_steps: int) -> tuple[Tensor, Tensor]:

        style_vector = self.sampler(
            noise,
            embedding=h_bert[0].unsqueeze(0),
            embedding_scale=embedding_scale,
            num_steps=diffusion_steps
        ).squeeze(0)

        # Split Style Vector
        style_vector_acoustic = style_vector[:, 128:]  # Right Half = Acoustic Style Vector
        style_vector_prosodic = style_vector[:, :128]  # Left Half = Prosodic Style Vector

        return style_vector_acoustic, style_vector_prosodic

    @torch.no_grad()
    def run_prediction(
        self,
        input_lengths: Tensor,
        text_mask: Tensor,
        h_bert: Tensor,
        h_text: Tensor,
        style_acoustic: Tensor,
        style_prosodic: Tensor
    ):
        # AdaIN injection
        h_bert_with_style = self.model.predictor.text_encoder(
            h_bert, style_acoustic, input_lengths, text_mask
        )

        # duration
        a_pred = self.predictDuration(h_bert_with_style, input_lengths)

        # align
        h_aligned = h_text @ a_pred.unsqueeze(0).to(self.device)

        # per-frame text embedding
        h_bert_per_frame = (h_bert_with_style.transpose(-1, -2) @ a_pred.unsqueeze(0).to(self.device))

        # pitch + energy
        f0_pred, n_pred = self.model.predictor.F0Ntrain(h_bert_per_frame, style_acoustic)

        return InferenceResult(
            h_text=h_text,
            h_aligned=h_aligned,
            f0_pred=f0_pred,
            a_pred=a_pred,
            n_pred=n_pred,
            style_vector_prosodic=style_prosodic
        )


    @torch.no_grad()
    def extract_embeddings(self, tokens: Tensor):
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(tokens.device)
        text_mask = length_to_mask(input_lengths).to(tokens.device)

        # Acoustic text encoder
        h_text = self.model.text_encoder(tokens, input_lengths, text_mask)

        # Prosodic BERT encoders
        h_bert_raw = self.model.bert(tokens, attention_mask=(~text_mask).int())
        h_bert = self.model.bert_encoder(h_bert_raw).transpose(-1, -2)

        return h_text, h_bert_raw, h_bert, input_lengths, text_mask

    @torch.no_grad()
    def inference_after_interpolation(self, input_lengths: Tensor, text_mask: Tensor, h_bert: Tensor, h_text: Tensor, style_vector_acoustic: Tensor, style_vector_prosodic: Tensor) -> InferenceResult:

        # AdaIN, Adding information of style vector to phoneme
        h_bert_with_style = self.model.predictor.text_encoder(h_bert, style_vector_acoustic, input_lengths, text_mask)

        # Function Call
        a_pred = self.predictDuration(h_bert_with_style, input_lengths)

        # Multiply alignment matrix with h_text
        h_aligned = h_text @ a_pred.unsqueeze(0).to(self.device)

        # Multiply per-phoneme embedding (bert_encoder_with_style) with frame-per-phoneme matrix -> per-frame text embedding
        h_bert_with_style_per_frame = (h_bert_with_style.transpose(-1, -2) @ a_pred.unsqueeze(0).to(self.device))

        f0_pred, n_pred = self.model.predictor.F0Ntrain(h_bert_with_style_per_frame, style_vector_acoustic)

        inferenceResult = InferenceResult(
            h_text=h_text,
            h_aligned=h_aligned,
            f0_pred=f0_pred,
            a_pred=a_pred,
            n_pred=n_pred,
            style_vector_prosodic=style_vector_prosodic,
        )

        return inferenceResult

    @torch.no_grad()
    def inference(self, tokens: Tensor) -> InferenceResult:
        # Number of phoneme / Length of tokens, shape[-1] = last element in list/array
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(tokens.device)

        # Creates a bitmask based on number of phonemes
        text_mask = length_to_mask(input_lengths).to(tokens.device)

        # Creates acoustic text encoder (phoneme -> feature vectors)
        h_text = self.model.text_encoder(tokens, input_lengths, text_mask)

        # Creates prosodic text encoder (phoneme -> feature vectors)
        h_bert_original = self.model.bert(tokens, attention_mask=(~text_mask).int())
        h_bert = self.model.bert_encoder(h_bert_original).transpose(-1, -2)

        # Function Call
        style_vector_acoustic, style_vector_prosodic = self.computeStyleVector(noise_gt, h_bert_original, embedding_scale, diffusion_steps)

        # AdaIN, Adding information of style vector to phoneme
        h_bert_with_style = self.model.predictor.text_encoder(h_bert, style_vector_acoustic, input_lengths, text_mask)

        ## Function Call
        a_pred = self.predictDuration(h_bert_with_style, input_lengths)

        # Multiply alignment matrix with h_text
        h_aligned = h_text @ a_pred.unsqueeze(0).to(self.device)  # (B, D_text, T_frames)

        # Multiply per-phoneme embedding (bert_encoder_with_style) with frame-per-phoneme matrix -> per-frame text embedding
        h_bert_with_style_per_frame = (h_bert_with_style.transpose(-1, -2) @ a_pred.unsqueeze(0).to(self.device))

        f0_pred, n_pred = self.model.predictor.F0Ntrain(h_bert_with_style_per_frame, style_vector_acoustic)

        inferenceResult = InferenceResult(
            h_text=h_text,
            h_aligned=h_aligned,
            f0_pred=f0_pred,
            a_pred=a_pred,
            n_pred=n_pred,
            style_vector_prosodic=style_vector_prosodic,
        )

        return inferenceResult

    @torch.no_grad()
    def synthesizeSpeech(self, inferenceResult: InferenceResult):

        out = self.model.decoder(
            inferenceResult.h_aligned,
            inferenceResult.f0_pred,
            inferenceResult.n_pred,
            inferenceResult.style_vector_prosodic.squeeze().unsqueeze(0)
        )

        return out.squeeze().cpu().numpy()

## Main Function

### Load Models

In [14]:
tts_model = StyleTTS2()
tts_model.load_models()  # builds self.model and loads self.params
tts_model.load_checkpoints()  # puts params into self.model
tts_model.sample_diffusion()  # builds self.sampler

embedding_model = TextEmbeddingModel(model_name="sentence-transformers/all-mpnet-base-v2")

In [15]:
asr_model = AutomaticSpeechRecognitionModel("tiny")

100%|█████████████████████████████████████| 72.1M/72.1M [00:05<00:00, 13.3MiB/s]


### Initializes Values

In [36]:
diffusion_steps = 5
embedding_scale = 1

interpolation_percentage = 0.4 # How much of Target to be used, small interpolation_percentage means more of ground_truth (Minimization)
to_interpolate = "h_text"

name_gt = "ground_truth"
text_gt = "I think the NFL is lame and boring"

name_target = "target"
text_target = "The Seattle Seahawks are the best Team in the world"

noise_gt = torch.randn(1, 1, 256).to(tts_model.device)
noise_target = torch.randn(1, 1, 256).to(tts_model.device)

### Inference with for-loop

#### Inference before interpolation

In [43]:
tokens_gt = tts_model.preprocessText(text_gt)
tokens_target = tts_model.preprocessText(text_target)

tokens_gt = addNumbersPattern(tokens_gt, tokens_target, [16, 4])
assert tokens_gt.shape == tokens_target.shape, "Padding didn't work, ground truth and target are of different dimensions"

h_text_gt, h_bert_raw_gt, h_bert_gt, input_lengths, text_mask = tts_model.extract_embeddings(tokens_gt)
h_text_target, h_bert_raw_target, h_bert_target, _, _ = tts_model.extract_embeddings(tokens_target)

h_text_mixed, h_bert_raw_mixed, h_bert_mixed = h_text_gt, h_bert_raw_gt, h_bert_gt

style_vector_acoustic, style_vector_prosodic = tts_model.computeStyleVector(noise_target, h_bert_raw_target, embedding_scale, diffusion_steps)

text_embedding_gt = embedding_model.encode(text_gt)
text_embedding_target = embedding_model.encode(text_target)

#### Initialize Optimizer

In [44]:
# Genetic algorithm parameters
algo_params = {
    "pop_size": 10,
}

num_generations = 10
num_objectives = 4
solution_shape = (input_lengths,)  # 16 decision variables per individual

optimizer = PymooOptimizer(
    bounds=(0, 1),
    algorithm=NSGA2,  # pass the class, not NSGA2(...)
    algo_params=algo_params,
    num_objectives=num_objectives,
    solution_shape=solution_shape,
)

#### For-Loop

##### Interpolate

In [46]:
for gen in range(num_generations):

    f1_list, f2_list, f3_list, f4_list = [], [], [], []

    # 1) Get current population
    population_vectors = optimizer.get_x_current()  # shape: (pop_size, *solution_shape)

    for i, interpolation_vector in enumerate(population_vectors):

        interpolation_vector = torch.from_numpy(interpolation_vector).to(h_text_gt.device).float()


        # Interpolate Values
        h_text_mixed = interpolateWithVector(h_text_gt, h_text_target, interpolation_vector)
        h_bert_mixed = interpolateWithVector(h_bert_gt, h_bert_target, interpolation_vector)

        inference_mixed = tts_model.inference_after_interpolation(
            input_lengths,
            text_mask,
            h_bert_mixed,
            h_text_mixed,
            style_vector_acoustic,
            style_vector_prosodic
        )

        audio_mixed = tts_model.synthesizeSpeech(inference_mixed)

        asr_result, asr_confidence = asr_model.analyzeAudio(audio_mixed)
        asr_text = asr_result["text"]

        text_embedding_mixed = embedding_model.encode(asr_text)

        # f1: Minimize Interpolation Vector (L1-Norm)
        f1 = float(interpolation_vector.abs().mean().item())

        # f2: Maximize Embedding Distance between ground-truth and ASR (cosine distance)
        f2 = - cosine_distance(text_embedding_gt, text_embedding_mixed)

        # f3: Minimize Embedding Distance between target and ASR (cosine distance)
        f3 = cosine_distance(text_embedding_target, text_embedding_mixed)

        # f4: Maximize ASR Confidence (Not negative, since its already a negative value)
        f4 = - asr_confidence

        f1_list.append(f1)
        f2_list.append(f2)
        f3_list.append(f3)
        f4_list.append(f4)

    pop_size = len(f1_list)
    dummy_data = [None] * pop_size

    optimizer.assign_fitness([f1_list, f2_list, f3_list, f4_list], dummy_data)
    optimizer.update()


    print(f"=== Generation {gen} ===")

    for idx, cand in enumerate(optimizer.best_candidates):
        print(f"Pareto Candidate {idx}:")
        print("Fitness vector:", cand.fitness)   # [f1, f2, f3, f4]
        print()

=== Generation 0 ===
Pareto Candidate 0:
Fitness vector: [ 0.5075618  -0.96801399  0.97281163  0.60854703]

=== Generation 1 ===
Pareto Candidate 0:
Fitness vector: [ 0.5075618  -0.96801399  0.97281163  0.60854703]

=== Generation 2 ===
Pareto Candidate 0:
Fitness vector: [ 0.5075618  -0.96801399  0.97281163  0.60854703]

=== Generation 3 ===
Pareto Candidate 0:
Fitness vector: [ 0.49221191 -0.94487443  0.78194752  0.70912631]

=== Generation 4 ===
Pareto Candidate 0:
Fitness vector: [ 0.49221191 -0.94487443  0.78194752  0.70912631]

=== Generation 5 ===
Pareto Candidate 0:
Fitness vector: [ 0.49221191 -0.94487443  0.78194752  0.70912631]

=== Generation 6 ===
Pareto Candidate 0:
Fitness vector: [ 0.49221191 -0.94487443  0.78194752  0.70912631]

=== Generation 7 ===
Pareto Candidate 0:
Fitness vector: [ 0.49221191 -0.94487443  0.78194752  0.70912631]

=== Generation 8 ===
Pareto Candidate 0:
Fitness vector: [ 0.50874096 -0.96123214  0.92816768  0.55068368]

=== Generation 9 ===
Pareto 

#### Inference after Interpolation

In [47]:
import IPython.display as ipd
import torch

print("=== Synthesis of Best Candidate After Optimization ===")

# 1) Get best candidate(s) from optimizer
best_candidates = optimizer.best_candidates

if len(best_candidates) == 0:
    raise RuntimeError("Optimizer has no best candidates. Did training run correctly?")

best = best_candidates[0]   # first Pareto-optimal candidate


# 2) Print FITNESS values
f1, f2, f3, f4 = best.fitness
print("Best candidate fitness values:")
print(f"  f1 (L1 norm)                = {f1}")
print(f"  f2 (maximize GT distance)   = {f2}")
print(f"  f3 (minimize target dist.)  = {f3}")
print(f"  f4 (maximize ASR conf.)     = {f4}")
print()


# 3) Extract interpolation vector
best_vector = torch.tensor(best.solution, device=tts_model.device).float()

# *** NEW: Print average value of the interpolation vector ***
avg_interp = best_vector.mean().item()
print(f"\nAverage interpolation value: {avg_interp:.6f}\n")


# 4) Create mixed embeddings using the best vector
if to_interpolate == "h_text":
    h_text_mixed_best = interpolateWithVector(h_text_gt, h_text_target, best_vector)
    h_bert_mixed_best = h_bert_gt

elif to_interpolate == "h_bert":
    h_text_mixed_best = h_text_gt
    h_bert_mixed_best = interpolateWithVector(h_bert_gt, h_bert_target, best_vector)

else:
    raise ValueError(f"Unknown interpolation target: {to_interpolate}")


# 5) Run TTS inference
with torch.no_grad():
    inference_best = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_mixed_best,
        h_text_mixed_best,
        style_vector_acoustic,
        style_vector_prosodic
    )
    audio_best = tts_model.synthesizeSpeech(inference_best)


# 6) Run ASR on final audio and print predicted text
asr_final, conf_final = asr_model.analyzeAudio(audio_best)
predicted_text_final = asr_final["text"].strip()

print("ASR Transcription of Best Candidate:")
print(f"  \"{predicted_text_final}\"")
print(f"  ASR confidence (avg_logprob) = {conf_final}")
print()


# 7) Play audio
display(ipd.Audio(audio_best, rate=24000))

print("Audio synthesis complete.")

=== Synthesis of Best Candidate After Optimization ===
Best candidate fitness values:
  f1 (L1 norm)                = 0.5087409615516663
  f2 (maximize GT distance)   = -0.9612321420748318
  f3 (minimize target dist.)  = 0.9281676784157753
  f4 (maximize ASR conf.)     = 0.5506836771965027


Average interpolation value: 0.508741

ASR Transcription of Best Candidate:
  "I see those nerf foamoseyer lame babyäm"
  ASR confidence (avg_logprob) = -3.9982147216796875



Audio synthesis complete.


### Write files

In [12]:
folder_path = "outputs/audio/padding_empty_phonemes/16_4_pattern/"

os.makedirs(folder_path, exist_ok=True)

sf.write(folder_path + "ground_truth.wav", audio_gt, samplerate=24000)
sf.write(folder_path + "target.wav", audio_target, samplerate=24000)
sf.write(folder_path + "interpolated.wav", audio_mixed, samplerate=24000)